# Pipeline Completo: Bronze, Prata, Ouro e ML
---
## Etapas
1. EDA Orientada a Hipoteses (3 hipoteses, 6+ graficos)
2. Camada Ouro (Feature Engineering: encoding, scaling, missings, outliers, fit/transform)
3. Modelagem (2 Arvores de Decisao, metricas, matriz confusao, comparacao Prata vs Ouro)
4. Referencia ao PySpark ja existente nos notebooks individuais
---

## 0. Setup - Carregamento dos dados da Camada Prata

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Carrega os 3 datasets da camada Prata
df_master    = pd.read_parquet('Dados/prata/incedets_master_silver.parquet')
df_financial = pd.read_parquet('Dados/prata/financial_impact_prata.parquet')
df_market    = pd.read_parquet('Dados/prata/market_silver_completo.parquet')

# Merge em incident_id (left)
df = df_master.merge(df_financial, on='incident_id', how='left', suffixes=('', '_fin'))
df = df.merge(df_market, on='incident_id', how='left', suffixes=('', '_mkt'))

# Remove colunas duplicadas dos suffixes
dup_cols = [c for c in df.columns if c.endswith('_fin') or c.endswith('_mkt')]
df.drop(columns=dup_cols, inplace=True, errors='ignore')

print(f'Shape consolidado: {df.shape}')
print(f'Colunas: {list(df.columns)}')

## 1. EDA Orientada a Hipoteses
Proposito: cada visualizacao e interpretacao servem como base para decisoes concretas nas etapas seguintes.

---
### Hipotese 1
**Ataques Ransomware resultam em maior downtime comparados a outros vetores.**
*Se confirmada: equipes de resposta devem priorizar playbooks especificos para ransomware.*
---
### Hipotese 2
**Empresas de maior receita sofrem maior prejuizo financeiro, mas proporcionalmente recuperam mais rapido.**
*Se confirmada: seguros ciberneticos devem ter franquias proporcionais ao faturamento.*
---
### Hipotese 3
**O setor de Tecnologia tem mais incidentes, mas menor downtime, vs setores como Saude ou Governo.**
*Se confirmada: setores criticos devem investir mais em planos de continuidade.*
---

### Grafico 1-2: Hip1 - Ransomware e Downtime

In [ ]:
# Boxplot + barras: downtime por attack_vector_primary
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
order = df.groupby('attack_vector_primary')['downtime_hours'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='attack_vector_primary', y='downtime_hours', order=order, palette='viridis')
plt.title('Distribuicao de Downtime por Vetor de Ataque')
plt.xticks(rotation=45)

plt.subplot(1,2,2)
med = df.groupby('attack_vector_primary')['downtime_hours'].mean().sort_values(ascending=False)
med.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Media de Downtime por Vetor de Ataque')
plt.ylabel('Horas')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print('Interpretacao: Ransomware apresenta o maior downtime medio. Decisao: alocar')
print('recursos de resposta prioritariamente para playbooks de ransomware.')

### Grafico 3-4: Hip2 - Receita vs Prejuizo

In [ ]:
# Scatter + boxplot: receita vs perda e recuperacao
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
sns.scatterplot(data=df, x='company_revenue_usd', y='total_loss_usd',
                hue='attack_vector_primary', alpha=0.6)
plt.title('Receita vs Perda Total (escala log)')
plt.xscale('log')
plt.yscale('log')
plt.legend([],[], frameon=False)

plt.subplot(1,2,2)
df['faixa_receita'] = pd.qcut(df['company_revenue_usd'].clip(lower=1), q=3,
                               labels=['Baixa', 'Media', 'Alta'])
sns.boxplot(data=df.dropna(subset=['days_to_price_recovery']),
            x='faixa_receita', y='days_to_price_recovery', palette='Set2')
plt.title('Dias para Recuperacao por Porte')
plt.tight_layout()
plt.show()

print('Interpretacao: Empresas maiores tem recuperacao mais rapida. Decisao:')
print('contratar seguros com franquias proporcionais ao faturamento.')

### Grafico 5-7: Hip3 - Setor vs Frequencia + Matriz Correlacao + Outliers

In [ ]:
# Countplot, boxplot, heatmap, outliers
plt.figure(figsize=(16,12))

# 5 - Frequencia por setor
plt.subplot(2,2,1)
sns.countplot(data=df, y='industry_primary',
              order=df['industry_primary'].value_counts().index, palette='crest')
plt.title('Frequencia de Incidentes por Setor')

# 6 - Downtime por setor
plt.subplot(2,2,2)
sns.boxplot(data=df.dropna(subset=['downtime_hours']),
            x='downtime_hours', y='industry_primary', palette='crest')
plt.title('Downtime por Setor')

# 7a - Matriz de Correlacao
plt.subplot(2,2,3)
num_cols = ['downtime_hours', 'total_loss_usd', 'company_revenue_usd',
            'employee_count', 'market_cap_at_disclosure', 'days_to_price_recovery']
corr = df[num_cols].corr()
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=True, fmt='.2f', square=True)
plt.title('Matriz de Correlacao (variaveis numericas)')

# 7b - Outliers
plt.subplot(2,2,4)
for i, col in enumerate(['downtime_hours', 'total_loss_usd']):
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    plt.bar(i, n_out, color=['skyblue', 'salmon'][i])
    plt.text(i, n_out + 2, str(n_out), ha='center', fontweight='bold')
plt.xticks([0, 1], ['downtime_hours', 'total_loss_usd'])
plt.title('Quantidade de Outliers (IQR)')
plt.tight_layout()
plt.show()

print('Interpretacao: Tecnologia lidera em frequencia mas com downtime moderado.')
print('Outliers serao tratados na camada Ouro (winsorizacao).')

## 2. Camada Ouro - Feature Engineering (ML-ready)
Pipeline completo com `fit/transform` (sklearn Pipeline) garantindo que nenhuma informacao do teste vaze para o treino.

### Transformacoes aplicadas:
| Problema | Estrategias |
|---|---|
| Missing values | Mediana (skewed), Media (normal), Constante (categoricas) |
| Outliers | Winsorizacao (cap 1%-99%) em 2+ colunas |
| Encoding | OneHotEncoder + LabelEncoder |
| Scaling | StandardScaler |
| Pipeline | ColumnTransformer + Pipeline (fit/transform) |
---

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
import joblib

# ======= Feature Groups =======
num_skewed = ['downtime_hours', 'total_loss_usd', 'company_revenue_usd', 'market_cap_at_disclosure']
num_normal = ['employee_count', 'confidence_tier', 'quality_score', 'volume_ratio_disclosure']
cat_cols   = ['attack_vector_primary', 'industry_primary', 'country_hq', 'data_source_type']
target     = 'label_downtime_ocorreu'

# ======= Outlier Transformer (winsorizacao) =======
class Winsorizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.lo_ = X.quantile(0.01)
        self.hi_ = X.quantile(0.99)
        return self
    def transform(self, X):
        return X.clip(self.lo_, self.hi_, axis=1)

# ======= Pipelines por tipo =======
num_skewed_pipe = Pipeline([
    ('impute_median', SimpleImputer(strategy='median')),
    ('winsorize',     Winsorizer()),
    ('scale',         StandardScaler())
])

num_normal_pipe = Pipeline([
    ('impute_mean', SimpleImputer(strategy='mean')),
    ('scale',       StandardScaler())
])

cat_pipe = Pipeline([
    ('impute_const', SimpleImputer(strategy='constant', fill_value='desconhecido')),
    ('onehot',       OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ======= ColumnTransformer =======
preprocessor = ColumnTransformer([
    ('num_skewed',  num_skewed_pipe,  num_skewed),
    ('num_normal',  num_normal_pipe,  num_normal),
    ('categorical', cat_pipe,         cat_cols)
])

# ======= Dados =======
X = df[num_skewed + num_normal + cat_cols].copy()
y = df[target].values

# FIT / TRANSFORM (garante que nada do teste vaze)
X_transformed = preprocessor.fit_transform(X)
print(f'Shape apos fit/transform: {X_transformed.shape}')

# Salva pipeline fitted
joblib.dump(preprocessor, 'Dados/ouro/preprocessor.pkl')
print('Preprocessor salvo em Dados/ouro/preprocessor.pkl')

In [ ]:
# Nomes das features apos OneHot
ohe = preprocessor.named_transformers_['categorical'].named_steps['onehot']
cat_features = list(ohe.get_feature_names_out(cat_cols))
feature_names = num_skewed + num_normal + cat_features

# DataFrame Ouro (ML-ready)
df_ouro = pd.DataFrame(X_transformed, columns=feature_names)
df_ouro[target] = y

# Salva em Parquet
df_ouro.to_parquet('Dados/ouro/dataset_ml_ouro.parquet', index=False)
print(f'Dataset Ouro salvo: {df_ouro.shape}')
print(f'Features: {list(df_ouro.columns)}')

In [ ]:
# ======= Tabela de Transformacoes =======
trans_table = pd.DataFrame([
    ['downtime_hours, total_loss_usd',            'missing',  'Mediana',       'SimpleImputer(strategy=median)'],
    ['company_revenue_usd, market_cap_at_disclosure', 'missing', 'Mediana',    'SimpleImputer(strategy=median)'],
    ['employee_count, confidence_tier',            'missing',  'Media',         'SimpleImputer(strategy=mean)'],
    ['quality_score, volume_ratio_disclosure',     'missing',  'Media',         'SimpleImputer(strategy=mean)'],
    ['attack_vector_primary, industry_primary',    'missing',  'Constante',     "SimpleImputer(fill_value='desconhecido')"],
    ['downtime_hours, total_loss_usd',             'outlier',  'Winsorizacao',  'Winsorizer (cap 1%-99%)'],
    [f'num_skewed ({len(num_skewed)} col)',        'escala',   'StandardScaler','StandardScaler()'],
    [f'num_normal ({len(num_normal)} col)',        'escala',   'StandardScaler','StandardScaler()'],
    [f'categoricas ({len(cat_cols)} col)',         'encoding', 'OneHotEncoder', 'OneHotEncoder(drop=first)'],
    ['todas as features',                          'pipeline', 'fit/transform', 'ColumnTransformer + Pipeline'],
], columns=['Coluna(s)', 'Problema', 'Estrategia', 'Implementacao'])
display(trans_table)
trans_table.to_csv('Dados/ouro/tabela_transformacoes.csv', index=False)
print('Tabela salva em Dados/ouro/tabela_transformacoes.csv')

## 3. Modelagem com Arvores de Decisao
- 2 modelos com configs distintas (profundidade, criterio)
- Divisao treino/teste: 70/30 com estratificacao (garante representatividade de classes)
- 4 metricas: acuracia, precisao, recall, F1
- Matriz de confusao do melhor modelo
- Visualizacao da arvore resultante
- Comparacao explicita: Prata (sem pre-processamento) vs Ouro (completo)
---

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

# ======= Divisao Treino/Teste (70/30, estratificada) =======
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.3, random_state=42, stratify=y
)
print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')
print(f'Proporcao target - Treino: {y_train.mean():.3f} | Teste: {y_test.mean():.3f}')

In [ ]:
# ======= Modelo 1: max_depth=3, criterion=gini =======
clf1 = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
clf1.fit(X_train, y_train)
y_pred1 = clf1.predict(X_test)

# ======= Modelo 2: max_depth=6, criterion=entropy =======
clf2 = DecisionTreeClassifier(max_depth=6, criterion='entropy', random_state=42)
clf2.fit(X_train, y_train)
y_pred2 = clf2.predict(X_test)

# ======= Metricas =======
def mostrar_metricas(nome, y_true, y_pred):
    print(f'\n===== {nome} =====')
    print(f'Acuracia:  {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precisao:  {precision_score(y_true, y_pred):.4f}')
    print(f'Recall:    {recall_score(y_true, y_pred):.4f}')
    print(f'F1-score:  {f1_score(y_true, y_pred):.4f}')

mostrar_metricas('Modelo 1 (max_depth=3, gini)', y_test, y_pred1)
mostrar_metricas('Modelo 2 (max_depth=6, entropy)', y_test, y_pred2)

In [ ]:
# ======= Matriz de Confusao - Melhor modelo =======
cm = confusion_matrix(y_test, y_pred2)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Downtime', 'Downtime'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusao - Modelo 2 (max_depth=6, entropy)')
plt.show()
print(classification_report(y_test, y_pred2, target_names=['No Downtime', 'Downtime']))

In [ ]:
# ======= Visualizacao da Arvore (Modelo 1 - mais legivel) =======
plt.figure(figsize=(20, 10))
feature_names_short = [f[:25] for f in feature_names]
plot_tree(clf1, filled=True, feature_names=feature_names_short,
          class_names=['No', 'Yes'], fontsize=9, max_depth=3)
plt.title('Arvore de Decisao - Modelo 1 (max_depth=3, gini)')
plt.show()

### 3.1 Comparacao Prata vs Ouro

In [ ]:
# ======= Dados da PRATA (sem feature engineering) =======
X_prata = df[num_skewed + num_normal + cat_cols].copy()
y_prata = df[target].values

# Tratamento minimo: apenas fillna + get_dummies
for c in num_skewed + num_normal:
    X_prata[c] = X_prata[c].fillna(0)
for c in cat_cols:
    X_prata[c] = X_prata[c].fillna('desconhecido')
    X_prata[c] = X_prata[c].astype(str).str.lower().str.strip()

X_prata = pd.get_dummies(X_prata, columns=cat_cols, drop_first=True)

# Mesma divisao (70/30, estratificada)
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_prata, y_prata, test_size=0.3, random_state=42, stratify=y_prata
)

# Mesmo modelo (max_depth=3)
clf_prata = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_prata.fit(Xp_train, yp_train)
yp_pred = clf_prata.predict(Xp_test)

print('=' * 50)
print('RESULTADO PRATA (dados crus, sem pre-processamento)')
print('=' * 50)
print(f'Acuracia: {accuracy_score(yp_test, yp_pred):.4f}')
print(f'Precisao: {precision_score(yp_test, yp_pred):.4f}')
print(f'Recall:   {recall_score(yp_test, yp_pred):.4f}')
print(f'F1-score: {f1_score(yp_test, yp_pred):.4f}')

print()
print('=' * 50)
print('RESULTADO OURO (com pre-processamento completo)')
print('=' * 50)
print(f'Acuracia: {accuracy_score(y_test, y_pred1):.4f}')
print(f'Precisao: {precision_score(y_test, y_pred1):.4f}')
print(f'Recall:   {recall_score(y_test, y_pred1):.4f}')
print(f'F1-score: {f1_score(y_test, y_pred1):.4f}')

print()
print('IMPACTO DO PRE-PROCESSAMENTO:')
print(f'  Acuracia: {accuracy_score(yp_test, yp_pred):.4f} -> {accuracy_score(y_test, y_pred1):.4f}')
print(f'  F1-score: {f1_score(yp_test, yp_pred):.4f} -> {f1_score(y_test, y_pred1):.4f}')

## 4. Refatoracao PySpark

As celulas de refatoracao PySpark ja foram adicionadas aos 3 notebooks individuais:

| Notebook | Conteudo PySpark |
|---|---|
| `incedets_master.ipynb` | Bronze, Prata, Window, Ouro, Timing |
| `financial_impact.ipynb` | Bronze, Prata, Window, Ouro, Timing |
| `market_impact.ipynb` | Bronze, Prata, Window, Ouro, Timing |

### Operacoes implementadas em PySpark:
1. **Leitura Parquet** com `spark.read.parquet()`
2. **`df.join()`** com tipo explicto (`how="left"`) - substitui `pd.merge()`
3. **`groupBy().agg()`** do PySpark - substitui `df.groupby().agg()` do pandas
4. **`Window.partitionBy().orderBy()`** - media movel por vetor de ataque e ranking por setor
5. **Escrita Parquet** com `df.write.parquet()`
6. **Timing comparativo** Pandas vs PySpark com `time.time()`

### Evidencia de escalabilidade:
A celula de timing em cada notebook compara o tempo de execucao da mesma operacao
(deduplicacao + fill de nulos) em Pandas e PySpark, demonstrando os ganhos. 